## Linear algebra method for determining an elementary flux mode

In [2]:

import numpy as np
import scipy.linalg as sla

Write down the stoichiometric matrix (S) for the network

In [8]:
S = np.array([
    [ 1, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0],
    [ 0,  1, -1,  0,  0,  0,  0,  0,  0,  0,  0],
    [ 0,  0,  1, -1,  0,  0,  0,  0,  0,  0,  0],
    [ 0,  0,  0,  1, -1,  0,  0,  0,  0,  0,  0],
    [ 0,  0,  0,  1,  1, -1,  0,  0,  0,  0,  0],
    [ 0,  0,  0,  0,  0,  1, -1,  0,  0,  0,  0],
    [ 0,  0,  0,  0,  0,  0,  1, -1,  0,  0,  0],
    [ 0,  0,  0,  0,  0,  0,  0,  1, -1,  0,  0],
    [-1,  0,  0,  0,  0,  0,  0,  0,  1, -1,  0],
    [ 1,  0,  0,  0,  0,  0,  0,  0,  0,  1, -1],
    [ 0,  0,  0,  0,  0,  1,  0,  0,  0,  0, -1],
    [ 0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  1],
], dtype=float)

print(S.shape)

(12, 11)


To determine the number of EFMs in this network, calculate the rank (= the number of independent variables) and the number of columns (= the number of reactions) of the stoichiometric matrix.

In [4]:
rank = np.linalg.matrix_rank(S)
n_reactions = S.shape[1]

print(f"Rank of S: {rank}")
print(f"Number of reactions: {n_reactions}")
print(f"Number of EFMs: {n_reactions - rank}")

Rank of S: 10
Number of reactions: 11
Number of EFMs: 1


If there is only a single EFM, the null space is a vector that corresponds to the flux distribution of this EFM.

In [5]:
N = sla.null_space(S)
print("\nNull space (EFMs):")
print(N)


Null space (EFMs):
[[-0.19611614]
 [-0.19611614]
 [-0.19611614]
 [-0.19611614]
 [-0.19611614]
 [-0.39223227]
 [-0.39223227]
 [-0.39223227]
 [-0.39223227]
 [-0.19611614]
 [-0.39223227]]


Verify that this is a steady state flux vector.

In [6]:
print(np.allclose(S @ N, 0))       # True
print(np.max(np.abs(S @ N)))        # Should be ~1e-15

True
2.220446049250313e-16


To determine the conserved moieties, the left null space of the stoichiometric matrix can be determined.

In [7]:
L = sla.null_space(S.T)
print("Number of conserved moieties:", L.shape[1])
print("Conserved moiety vectors (columns):\n", L)

Number of conserved moieties: 2
Conserved moiety vectors (columns):
 [[ 1.44110736e-16 -5.63991807e-18]
 [ 1.00833050e-16  0.00000000e+00]
 [ 6.89612979e-17  6.68423175e-17]
 [ 1.24786878e-17  1.57740864e-17]
 [ 5.81615890e-17 -5.38055572e-17]
 [ 4.26401433e-01  5.04328969e-17]
 [ 4.26401433e-01  2.33242962e-17]
 [ 4.26401433e-01  6.84613325e-17]
 [ 4.26401433e-01  3.74414580e-17]
 [ 4.26401433e-01  4.30813761e-17]
 [-2.13200716e-01  7.07106781e-01]
 [ 2.13200716e-01  7.07106781e-01]]


In [9]:
from sympy import Matrix

S_sym = Matrix(S.astype(int).tolist())

# Left null space = null space of S.T
left_null = S_sym.T.nullspace()

print(f"Number of conserved moieties: {len(left_null)}")
for i, v in enumerate(left_null):
    print(f"\nMoiety {i+1}: {v.T}")

Number of conserved moieties: 2

Moiety 1: Matrix([[0, 0, 0, 0, 0, -1, -1, -1, -1, -1, 1, 0]])

Moiety 2: Matrix([[0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1]])


In [10]:
species = ['G6P', 'F6P', 'F16BP', 'DHAP', 'G3P', 'BPG', 'PG3', 'PG2', 'PEP', 'PYR', 'NADH', 'NAD']

for i, v in enumerate(left_null):
    coeffs = list(v)
    moiety = " + ".join(
        f"{c}·{s}" for c, s in zip(coeffs, species) if c != 0
    )
    print(f"Conserved moiety {i+1}: {moiety} = constant")

Conserved moiety 1: -1·BPG + -1·PG3 + -1·PG2 + -1·PEP + -1·PYR + 1·NADH = constant
Conserved moiety 2: 1·BPG + 1·PG3 + 1·PG2 + 1·PEP + 1·PYR + 1·NAD = constant
